In [1]:
!pip install duckdb

In [2]:
from google.colab import files

uploaded = files.upload()

Saving olist_customers_dataset.csv to olist_customers_dataset.csv
Saving olist_geolocation_dataset.csv to olist_geolocation_dataset.csv
Saving olist_order_items_dataset.csv to olist_order_items_dataset.csv
Saving olist_order_payments_dataset.csv to olist_order_payments_dataset.csv
Saving olist_order_reviews_dataset.csv to olist_order_reviews_dataset.csv
Saving olist_orders_dataset.csv to olist_orders_dataset.csv
Saving olist_products_dataset.csv to olist_products_dataset.csv
Saving olist_sellers_dataset.csv to olist_sellers_dataset.csv
Saving product_category_name_translation.csv to product_category_name_translation.csv


In [3]:
import os

os.listdir()

['.config',
 'olist_orders_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'product_category_name_translation.csv',
 'olist_order_payments_dataset.csv',
 'olist_products_dataset.csv',
 'olist_customers_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_sellers_dataset.csv',
 'olist_geolocation_dataset.csv',
 'sample_data']

In [4]:
import duckdb

con = duckdb.connect("olist_product_analytics.duckdb")

In [5]:
con.execute("""
CREATE OR REPLACE TABLE customers AS
SELECT *
FROM read_csv_auto('olist_customers_dataset.csv');
""")

In [6]:
con.execute("""
CREATE OR REPLACE TABLE orders AS
SELECT *
FROM read_csv_auto('olist_orders_dataset.csv');
""")

In [7]:
con.execute("""
CREATE OR REPLACE TABLE order_items AS
SELECT *
FROM read_csv_auto('olist_order_items_dataset.csv');
""")

In [8]:
con.execute("""
CREATE OR REPLACE TABLE order_payments AS
SELECT *
FROM read_csv_auto('olist_order_payments_dataset.csv');
""")

con.execute("""
CREATE OR REPLACE TABLE order_reviews AS
SELECT *
FROM read_csv_auto('olist_order_reviews_dataset.csv');
""")

con.execute("""
CREATE OR REPLACE TABLE products AS
SELECT *
FROM read_csv_auto('olist_products_dataset.csv');
""")

con.execute("""
CREATE OR REPLACE TABLE sellers AS
SELECT *
FROM read_csv_auto('olist_sellers_dataset.csv');
""")

con.execute("""
CREATE OR REPLACE TABLE geolocation AS
SELECT *
FROM read_csv_auto('olist_geolocation_dataset.csv');
""")

con.execute("""
CREATE OR REPLACE TABLE category_translation AS
SELECT *
FROM read_csv_auto('product_category_name_translation.csv');
""")

In [9]:
con.execute("SHOW TABLES").fetchdf()

,name
0,category_translation
1,customers
2,geolocation
3,order_items
4,order_payments
5,order_reviews
6,orders
7,products
8,sellers


In [10]:
con.execute("""
SELECT *
FROM customers
LIMIT 5
""").fetchdf()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [11]:
tables = [
    "customers",
    "orders",
    "order_items",
    "order_payments",
    "order_reviews",
    "products",
    "sellers",
    "geolocation",
    "category_translation"
]

for table in tables:
    count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(table, ":", count)

customers : 99441
orders : 99441
order_items : 112650
order_payments : 103886
order_reviews : 99224
products : 32951
sellers : 3095
geolocation : 1000163
category_translation : 71


In [12]:
con.execute("""
SELECT
    o.order_id,
    o.customer_id,
    c.customer_unique_id,
    o.order_status,
    o.order_purchase_timestamp,
    oi.product_id,
    oi.price,
    oi.freight_value
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
LIMIT 10;
""").fetchdf()

,order_id,customer_id,customer_unique_id,order_status,order_purchase_timestamp,product_id,price,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,delivered,2017-10-02 10:56:33,87285b34884572647811a353c7ac498a,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,delivered,2018-07-24 20:41:37,595fac2a385ac33a80bd5114aec74eb8,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,delivered,2018-08-08 08:38:49,aa4383b373c6aca5d8797843e5594415,159.90,19.22
3,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,delivered,2018-02-13 21:18:39,65266b2da20d04dbe00c5c2d3bb7859e,19.90,8.72
4,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,80bb27c7c16e8f973207a5086ab329e2,delivered,2017-07-09 21:57:05,060cb19345d90064d1015407193c233d,147.90,27.36
5,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,36edbb3fb164b1f16485364b6fb04c73,invoiced,2017-04-11 12:22:08,a1804276d9941ac0733cfd409f5206eb,49.90,16.05
6,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,932afa1e708222e5821dac9cd5db4cae,delivered,2017-05-16 13:10:30,4520766ec412348b8d4caa5e8a18c464,59.99,15.17
7,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,39382392765b6dc74812866ee5ee92a7,delivered,2017-01-23 18:29:09,ac1789e492dcd698c5c10b97a671243a,19.90,16.05
8,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,299905e3934e9e181bfb2e164dd4b4f8,delivered,2017-07-29 11:55:02,9a78fb9862b10749a117f7fc3c31f051,149.99,19.77
9,e6ce16cb79ec1d90b1da9085a6118aeb,494dded5b201313c64ed7f100595b95c,f2a85dec752b8517b5e58a06ff3cd937,delivered,2017-05-16 19:41:10,08574b074924071f4e201e151b152b4e,99.00,30.53


In [13]:
con.execute("""
CREATE OR REPLACE TABLE order_analytics AS

SELECT
    o.order_id,
    c.customer_unique_id,
    c.customer_city,
    c.customer_state,

    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,

    oi.product_id,
    oi.seller_id,
    oi.price,
    oi.freight_value,

    p.product_category_name,
    ct.product_category_name_english,

    r.review_score,

    op.payment_type,
    op.payment_installments,
    op.payment_value

FROM orders o

LEFT JOIN customers c
    ON o.customer_id = c.customer_id

LEFT JOIN order_items oi
    ON o.order_id = oi.order_id

LEFT JOIN products p
    ON oi.product_id = p.product_id

LEFT JOIN category_translation ct
    ON p.product_category_name = ct.product_category_name

LEFT JOIN order_reviews r
    ON o.order_id = r.order_id

LEFT JOIN order_payments op
    ON o.order_id = op.order_id;
""")

In [14]:
con.execute("""
SELECT *
FROM order_analytics
LIMIT 10
""").fetchdf()

,order_id,customer_unique_id,customer_city,customer_state,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,product_id,seller_id,price,freight_value,product_category_name,product_category_name_english,review_score,payment_type,payment_installments,payment_value
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,29.99,8.72,utilidades_domesticas,housewares,4,voucher,1,18.59
1,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,barreiras,BA,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,118.70,22.76,perfumaria,perfumery,4,boleto,1,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,159.90,19.22,automotivo,auto,5,credit_card,3,179.12
3,ad21c59c0840e6cb83a9ceb5573f8159,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,19.90,8.72,papelaria,stationery,5,credit_card,1,28.62
4,a4591c265e18cb1dcee52889e2d8acc3,80bb27c7c16e8f973207a5086ab329e2,congonhinhas,PR,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01,060cb19345d90064d1015407193c233d,8581055ce74af1daba164fdbd55a40de,147.90,27.36,automotivo,auto,4,credit_card,6,175.26
5,6514b8ad8028c9f2cc2374ded245783f,932afa1e708222e5821dac9cd5db4cae,nilopolis,RJ,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07,4520766ec412348b8d4caa5e8a18c464,16090f2ca825584b5a147ab24aa30c86,59.99,15.17,automotivo,auto,5,credit_card,3,75.16
6,76c6e866289321a7c93b82b54852dc33,39382392765b6dc74812866ee5ee92a7,faxinalzinho,RS,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06,ac1789e492dcd698c5c10b97a671243a,63b9ae557efed31d1f7687917d248a8d,19.90,16.05,moveis_decoracao,furniture_decor,1,boleto,1,35.95
7,e69bfb5eb88e0ed6a785585b27e16dbf,299905e3934e9e181bfb2e164dd4b4f8,sorocaba,SP,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23,9a78fb9862b10749a117f7fc3c31f051,7c67e1448b00f6e969d365cea6b010ab,149.99,19.77,moveis_escritorio,office_furniture,5,credit_card,1,8.34
8,e6ce16cb79ec1d90b1da9085a6118aeb,f2a85dec752b8517b5e58a06ff3cd937,rio de janeiro,RJ,delivered,2017-05-16 19:41:10,2017-05-16 19:50:18,2017-05-18 11:40:40,2017-05-29 11:18:31,2017-06-07,08574b074924071f4e201e151b152b4e,001cca7ae9ae17fb1caed9dfb1094831,99.00,30.53,ferramentas_jardim,garden_tools,1,credit_card,1,259.06
9,34513ce0c4fab462a55830c0989c7edb,782987b81c92239d922aa49d6bd4200b,sao paulo,SP,delivered,2017-07-13 19:58:11,2017-07-13 20:10:08,2017-07-14 18:43:29,2017-07-19 14:04:48,2017-08-08,f7e0fa615b386bc9a8b9eb52bc1fff76,87142160b41353c4e5fca2360caf6f92,98.00,16.13,informatica_acessorios,computers_accessories,4,credit_card,1,114.13


In [15]:
con.execute("""
SELECT COUNT(*)
FROM order_analytics
""").fetchdf()

,count_star()
0,119143


How healthy is the customer experience from purchase to delivery and review?

In [16]:
con.execute("""
SELECT
    order_status,
    COUNT(DISTINCT order_id) AS orders
FROM orders
GROUP BY order_status
ORDER BY orders DESC;
""").fetchdf()

,order_status,orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


Calculate the basic product KPIs

In [17]:
con.execute("""
SELECT
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT o.customer_id) AS total_customers,
    COUNT(DISTINCT c.customer_unique_id) AS unique_customers,
    SUM(oi.price) AS product_revenue,
    AVG(oi.price) AS avg_item_price
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id;
""").fetchdf()

,total_orders,total_customers,unique_customers,product_revenue,avg_item_price
0,98666,98666,95420,1.359164e+07,120.653739


In [18]:
con.execute("""
SELECT
    customer_unique_id,
    COUNT(DISTINCT order_id) AS number_of_orders
FROM order_analytics
GROUP BY customer_unique_id
ORDER BY number_of_orders DESC
LIMIT 20;
""").fetchdf()

,customer_unique_id,number_of_orders
0,8d50f5eadf50201ccdcedfb9e2ac8455,17
1,3e43e6105506432c953e165fb2acf44c,9
2,6469f99c1f9dfae7733b25662e7f1782,7
3,1b6c7548a2a1f9037c1fd3ddfed95f33,7
4,ca77025e7201e3b30c44b472ff346268,7
5,de34b16117594161a6a89c50b289d35a,6
6,12f5d6e1cbf93dafd9dcc19095df0b3d,6
7,63cfc61cee11cbe306bff5857d00bfe4,6
8,dc813062e0fc23409cd255f7f53c7074,6
9,f0e310a6839dce9de1638e0fe5ab282a,6


In [19]:
con.execute("""
WITH customer_orders AS (

    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS orders_count

    FROM orders o

    JOIN customers c
        ON o.customer_id = c.customer_id

    WHERE o.order_status = 'delivered'

    GROUP BY c.customer_unique_id
)

SELECT
    COUNT(*) AS total_customers,

    SUM(
        CASE
            WHEN orders_count >= 2 THEN 1
            ELSE 0
        END
    ) AS repeat_customers,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN orders_count >= 2 THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS repeat_customer_rate

FROM customer_orders;
""").fetchdf()

,total_customers,repeat_customers,repeat_customer_rate
0,93358,2801.0,3.0


Order-status funnel

In [20]:
con.execute("""
SELECT
    order_status,
    COUNT(DISTINCT order_id) AS orders
FROM orders
GROUP BY order_status
ORDER BY orders DESC;
""").fetchdf()

,order_status,orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


basic product kpis

In [21]:
con.execute("""
SELECT
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT o.customer_id) AS total_customers,
    COUNT(DISTINCT c.customer_unique_id) AS unique_customers,
    SUM(oi.price) AS product_revenue,
    AVG(oi.price) AS avg_item_price
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id;
""").fetchdf()

,total_orders,total_customers,unique_customers,product_revenue,avg_item_price
0,98666,98666,95420,1.359164e+07,120.653739


how much orders does each customer make

In [22]:
con.execute("""
SELECT
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id) AS number_of_orders
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
GROUP BY c.customer_unique_id
ORDER BY number_of_orders DESC
LIMIT 20;
""").fetchdf()

,customer_unique_id,number_of_orders
0,8d50f5eadf50201ccdcedfb9e2ac8455,17
1,3e43e6105506432c953e165fb2acf44c,9
2,6469f99c1f9dfae7733b25662e7f1782,7
3,1b6c7548a2a1f9037c1fd3ddfed95f33,7
4,ca77025e7201e3b30c44b472ff346268,7
5,12f5d6e1cbf93dafd9dcc19095df0b3d,6
6,63cfc61cee11cbe306bff5857d00bfe4,6
7,47c1a3033b8b77b3ab6e109eb4d5fdf3,6
8,de34b16117594161a6a89c50b289d35a,6
9,f0e310a6839dce9de1638e0fe5ab282a,6


repeat customer rate

In [23]:
con.execute("""
WITH customer_orders AS (

    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS orders_count

    FROM orders o

    JOIN customers c
        ON o.customer_id = c.customer_id

    WHERE o.order_status = 'delivered'

    GROUP BY c.customer_unique_id
)

SELECT
    COUNT(*) AS total_customers,

    SUM(
        CASE
            WHEN orders_count >= 2 THEN 1
            ELSE 0
        END
    ) AS repeat_customers,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN orders_count >= 2 THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS repeat_customer_rate

FROM customer_orders;
""").fetchdf()

,total_customers,repeat_customers,repeat_customer_rate
0,93358,2801.0,3.0


delivery delay.

In [24]:
con.execute("""
SELECT
    order_id,
    DATE_DIFF(
        'day',
        order_delivered_customer_date,
        order_estimated_delivery_date
    ) AS delivery_difference_days
FROM orders
WHERE order_status = 'delivered'
LIMIT 20;
""").fetchdf()

,order_id,delivery_difference_days
0,e481f51cbdc54678b7cc49136f2d6af7,8
1,53cdb2fc8bc7dce0b6741e2150273451,6
2,47770eb9100c2d0c44946d9cf07ec65d,18
3,949d5b44dbf5de918fe9c16f97b45f8a,13
4,ad21c59c0840e6cb83a9ceb5573f8159,10
5,a4591c265e18cb1dcee52889e2d8acc3,6
6,6514b8ad8028c9f2cc2374ded245783f,12
7,76c6e866289321a7c93b82b54852dc33,32
8,e69bfb5eb88e0ed6a785585b27e16dbf,7
9,e6ce16cb79ec1d90b1da9085a6118aeb,9


Classify every delivered order

In [26]:
con.execute("""
SELECT
    CASE
        WHEN order_delivered_customer_date < order_estimated_delivery_date
            THEN 'Early'
        WHEN order_delivered_customer_date = order_estimated_delivery_date
            THEN 'On Time'
        WHEN order_delivered_customer_date > order_estimated_delivery_date
            THEN 'Late'
    END AS delivery_performance,

    COUNT(*) AS orders

FROM orders

WHERE order_status = 'delivered'

GROUP BY delivery_performance

ORDER BY orders DESC;
""").fetchdf()

,delivery_performance,orders
0,Early,88644
1,Late,7826
2,None,8


In [27]:
result = con.execute("""
WITH delivered_orders AS (

    SELECT
        o.order_id,
        c.customer_unique_id,

        CASE
            WHEN o.order_delivered_customer_date
                 <= o.order_estimated_delivery_date
                THEN 'Early/On Time'

            WHEN o.order_delivered_customer_date
                 > o.order_estimated_delivery_date
                THEN 'Late'
        END AS delivery_performance

    FROM orders o

    JOIN customers c
        ON o.customer_id = c.customer_id

    WHERE o.order_status = 'delivered'
),

customer_delivery AS (

    SELECT
        customer_unique_id,

        MAX(
            CASE
                WHEN delivery_performance = 'Late'
                THEN 1
                ELSE 0
            END
        ) AS had_late_delivery

    FROM delivered_orders

    GROUP BY customer_unique_id
),

customer_orders AS (

    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS order_count

    FROM orders o

    JOIN customers c
        ON o.customer_id = c.customer_id

    WHERE o.order_status = 'delivered'

    GROUP BY c.customer_unique_id
)

SELECT
    CASE
        WHEN cd.had_late_delivery = 1
            THEN 'Had Late Delivery'
        ELSE 'No Late Delivery'
    END AS delivery_group,

    COUNT(*) AS customers,

    SUM(
        CASE
            WHEN co.order_count >= 2
            THEN 1
            ELSE 0
        END
    ) AS repeat_customers,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN co.order_count >= 2
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS repeat_customer_rate

FROM customer_delivery cd

JOIN customer_orders co
    ON cd.customer_unique_id = co.customer_unique_id

GROUP BY delivery_group
ORDER BY delivery_group;
""").fetchdf()

result

,delivery_group,customers,repeat_customers,repeat_customer_rate
0,Had Late Delivery,7771,358.0,4.61
1,No Late Delivery,85587,2443.0,2.85


In [28]:
result = con.execute("""
WITH ranked_orders AS (

    SELECT
        o.order_id,
        c.customer_unique_id,
        o.order_purchase_timestamp,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date,

        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY o.order_purchase_timestamp
        ) AS order_number

    FROM orders o

    JOIN customers c
        ON o.customer_id = c.customer_id

    WHERE o.order_status = 'delivered'
),

first_orders AS (

    SELECT
        customer_unique_id,
        order_id,

        CASE
            WHEN order_delivered_customer_date
                 <= order_estimated_delivery_date
                THEN 'On Time / Early'

            WHEN order_delivered_customer_date
                 > order_estimated_delivery_date
                THEN 'Late'
        END AS first_delivery_performance

    FROM ranked_orders

    WHERE order_number = 1
),

customer_order_counts AS (

    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS total_orders

    FROM orders o

    JOIN customers c
        ON o.customer_id = c.customer_id

    WHERE o.order_status = 'delivered'

    GROUP BY c.customer_unique_id
)

SELECT
    fo.first_delivery_performance,

    COUNT(*) AS customers,

    SUM(
        CASE
            WHEN coc.total_orders >= 2
            THEN 1
            ELSE 0
        END
    ) AS repeat_customers,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN coc.total_orders >= 2
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS repeat_customer_rate

FROM first_orders fo

JOIN customer_order_counts coc
    ON fo.customer_unique_id = coc.customer_unique_id

WHERE fo.first_delivery_performance IS NOT NULL

GROUP BY fo.first_delivery_performance
ORDER BY fo.first_delivery_performance;
""").fetchdf()

result

,first_delivery_performance,customers,repeat_customers,repeat_customer_rate
0,Late,7602,189.0,2.49
1,On Time / Early,85748,2612.0,3.05


In [29]:
con.execute("""
WITH first_orders AS (

    SELECT
        o.order_id,
        c.customer_unique_id,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date,

        CASE
            WHEN o.order_delivered_customer_date
                 <= o.order_estimated_delivery_date
                THEN 'On Time / Early'
            WHEN o.order_delivered_customer_date
                 > o.order_estimated_delivery_date
                THEN 'Late'
        END AS first_delivery_performance,

        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY o.order_purchase_timestamp
        ) AS order_number

    FROM orders o

    JOIN customers c
        ON o.customer_id = c.customer_id

    WHERE o.order_status = 'delivered'
),

first_order_reviews AS (

    SELECT
        fo.customer_unique_id,
        fo.first_delivery_performance,
        r.review_score

    FROM first_orders fo

    JOIN order_reviews r
        ON fo.order_id = r.order_id

    WHERE fo.order_number = 1
)

SELECT
    first_delivery_performance,
    COUNT(*) AS reviewed_orders,
    ROUND(AVG(review_score), 2) AS average_review_score
FROM first_order_reviews
WHERE first_delivery_performance IS NOT NULL
GROUP BY first_delivery_performance
ORDER BY first_delivery_performance;
""").fetchdf()

,first_delivery_performance,reviewed_orders,average_review_score
0,Late,7467,2.56
1,On Time / Early,85555,4.29


In [30]:
con.execute("""
WITH ranked_orders AS (
    SELECT
        o.order_id,
        c.customer_unique_id,
        o.order_purchase_timestamp,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date,

        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY o.order_purchase_timestamp
        ) AS order_number

    FROM orders o
    JOIN customers c
        ON o.customer_id = c.customer_id

    WHERE o.order_status = 'delivered'
),

first_orders AS (
    SELECT
        customer_unique_id,
        order_id,
        order_purchase_timestamp,

        CASE
            WHEN order_delivered_customer_date <= order_estimated_delivery_date
                THEN 'On Time / Early'
            WHEN order_delivered_customer_date > order_estimated_delivery_date
                THEN 'Late'
        END AS first_delivery_performance

    FROM ranked_orders
    WHERE order_number = 1
),

eligible_customers AS (
    SELECT *
    FROM first_orders
    WHERE order_purchase_timestamp
          <= (SELECT MAX(order_purchase_timestamp) FROM orders)
             - INTERVAL '90 days'
),

repeat_within_90_days AS (
    SELECT DISTINCT
        f.customer_unique_id

    FROM eligible_customers f

    JOIN ranked_orders r
        ON f.customer_unique_id = r.customer_unique_id

    WHERE r.order_number >= 2
      AND r.order_purchase_timestamp > f.order_purchase_timestamp
      AND r.order_purchase_timestamp
          <= f.order_purchase_timestamp + INTERVAL '90 days'
)

SELECT
    f.first_delivery_performance,
    COUNT(*) AS eligible_customers,

    COUNT(r.customer_unique_id) AS repeat_customers_90d,

    ROUND(
        100.0 * COUNT(r.customer_unique_id) / COUNT(*),
        2
    ) AS repeat_rate_90d

FROM eligible_customers f

LEFT JOIN repeat_within_90_days r
    ON f.customer_unique_id = r.customer_unique_id

WHERE f.first_delivery_performance IS NOT NULL

GROUP BY f.first_delivery_performance
ORDER BY f.first_delivery_performance;
""").fetchdf()

,first_delivery_performance,eligible_customers,repeat_customers_90d,repeat_rate_90d
0,Late,6807,122,1.79
1,On Time / Early,77564,1484,1.91


In [31]:
con.execute("""
SELECT
    CASE
        WHEN DATE_DIFF(
            'day',
            order_estimated_delivery_date,
            order_delivered_customer_date
        ) <= 3
            THEN '1-3 days late'

        WHEN DATE_DIFF(
            'day',
            order_estimated_delivery_date,
            order_delivered_customer_date
        ) <= 7
            THEN '4-7 days late'

        WHEN DATE_DIFF(
            'day',
            order_estimated_delivery_date,
            order_delivered_customer_date
        ) <= 14
            THEN '8-14 days late'

        ELSE '15+ days late'
    END AS delay_bucket,

    COUNT(*) AS orders,

    ROUND(
        AVG(
            DATE_DIFF(
                'day',
                order_estimated_delivery_date,
                order_delivered_customer_date
            )
        ),
        2
    ) AS avg_days_late

FROM orders

WHERE order_status = 'delivered'

  AND order_delivered_customer_date
      > order_estimated_delivery_date

GROUP BY delay_bucket

ORDER BY
    CASE delay_bucket
        WHEN '1-3 days late' THEN 1
        WHEN '4-7 days late' THEN 2
        WHEN '8-14 days late' THEN 3
        WHEN '15+ days late' THEN 4
    END;
""").fetchdf()

,delay_bucket,orders,avg_days_late
0,1-3 days late,3162,1.08
1,4-7 days late,1802,5.52
2,8-14 days late,1478,10.59
3,15+ days late,1384,29.18


In [32]:
con.execute("""
SELECT
    c.customer_state,

    COUNT(DISTINCT o.order_id) AS delivered_orders,

    COUNT(DISTINCT CASE
        WHEN o.order_delivered_customer_date
             > o.order_estimated_delivery_date
        THEN o.order_id
    END) AS late_orders,

    ROUND(
        100.0 *
        COUNT(DISTINCT CASE
            WHEN o.order_delivered_customer_date
                 > o.order_estimated_delivery_date
            THEN o.order_id
        END)
        / COUNT(DISTINCT o.order_id),
        2
    ) AS late_rate

FROM orders o

JOIN customers c
    ON o.customer_id = c.customer_id

WHERE o.order_status = 'delivered'

GROUP BY c.customer_state

HAVING COUNT(DISTINCT o.order_id) >= 500

ORDER BY late_rate DESC;
""").fetchdf()

,customer_state,delivered_orders,late_orders,late_rate
0,MA,717,141,19.67
1,CE,1279,196,15.32
2,BA,3256,457,14.04
3,RJ,12350,1664,13.47
4,PA,946,117,12.37
5,ES,1995,244,12.23
6,MS,701,81,11.55
7,PB,517,57,11.03
8,PE,1593,172,10.80
9,SC,3546,346,9.76


In [33]:
con.execute("""
SELECT
    oi.seller_id,

    COUNT(DISTINCT oi.order_id) AS delivered_orders,

    COUNT(DISTINCT CASE
        WHEN o.order_delivered_customer_date
             > o.order_estimated_delivery_date
        THEN oi.order_id
    END) AS late_orders,

    ROUND(
        100.0 *
        COUNT(DISTINCT CASE
            WHEN o.order_delivered_customer_date
                 > o.order_estimated_delivery_date
            THEN oi.order_id
        END)
        / COUNT(DISTINCT oi.order_id),
        2
    ) AS late_rate

FROM order_items oi

JOIN orders o
    ON oi.order_id = o.order_id

WHERE o.order_status = 'delivered'

GROUP BY oi.seller_id

HAVING COUNT(DISTINCT oi.order_id) >= 100

ORDER BY late_rate DESC;
""").fetchdf()

,seller_id,delivered_orders,late_orders,late_rate
0,06a2c3af7b3aee5d69171b0e14f0ee87,389,90,23.14
1,1ca7077d890b907f89be8c954a02686a,108,24,22.22
2,88460e8ebdecbfecb5f9601833981930,246,48,19.51
3,e5a3438891c0bfdb9394643f95273d8e,216,40,18.52
4,cd68562d3f44870c08922d380acae552,122,22,18.03
...,...,...,...,...
205,229c3efbfb0ea2058de4ccdfbc3d784a,118,2,1.69
206,12b9676b00f60f3b700e83af21824c0e,133,2,1.50
207,8c16d1f32a54d92897cc437244442e1b,108,1,0.93
208,0bae85eb84b9fb3bd773911e89288d54,136,1,0.74


In [34]:
con.execute("""
SELECT
    COALESCE(
        p.product_category_name,
        'Unknown'
    ) AS category,

    COUNT(DISTINCT o.order_id) AS delivered_orders,

    COUNT(DISTINCT CASE
        WHEN o.order_delivered_customer_date
             > o.order_estimated_delivery_date
        THEN o.order_id
    END) AS late_orders,

    ROUND(
        100.0 *
        COUNT(DISTINCT CASE
            WHEN o.order_delivered_customer_date
                 > o.order_estimated_delivery_date
            THEN o.order_id
        END)
        / COUNT(DISTINCT o.order_id),
        2
    ) AS late_rate

FROM orders o

JOIN order_items oi
    ON o.order_id = oi.order_id

JOIN products p
    ON oi.product_id = p.product_id

WHERE o.order_status = 'delivered'

GROUP BY category

HAVING COUNT(DISTINCT o.order_id) >= 200

ORDER BY late_rate DESC;
""").fetchdf()

,category,delivered_orders,late_orders,late_rate
0,audio,348,45,12.93
1,livros_tecnicos,256,28,10.94
2,casa_conforto,392,41,10.46
3,alimentos,441,44,9.98
4,eletronicos,2517,247,9.81
5,bebes,2809,258,9.18
6,moveis_escritorio,1254,115,9.17
7,Unknown,1392,127,9.12
8,construcao_ferramentas_construcao,736,67,9.10
9,construcao_ferramentas_iluminacao,242,22,9.09


In [35]:
con.execute("""
SELECT
    p.product_category_name AS category,
    oi.seller_id,

    COUNT(DISTINCT o.order_id) AS delivered_orders,

    COUNT(DISTINCT CASE
        WHEN o.order_delivered_customer_date
             > o.order_estimated_delivery_date
        THEN o.order_id
    END) AS late_orders,

    ROUND(
        100.0 *
        COUNT(DISTINCT CASE
            WHEN o.order_delivered_customer_date
                 > o.order_estimated_delivery_date
            THEN o.order_id
        END)
        / COUNT(DISTINCT o.order_id),
        2
    ) AS late_rate

FROM orders o

JOIN order_items oi
    ON o.order_id = oi.order_id

JOIN products p
    ON oi.product_id = p.product_id

WHERE o.order_status = 'delivered'

GROUP BY
    category,
    oi.seller_id

HAVING COUNT(DISTINCT o.order_id) >= 50

ORDER BY late_rate DESC
LIMIT 30;
""").fetchdf()

,category,seller_id,delivered_orders,late_orders,late_rate
0,cama_mesa_banho,54965bbe3e4f07ae045b90b0b8541f52,73,22,30.14
1,brinquedos,a49928bcdf77c55c6d6e05e09a9b4ca5,82,21,25.61
2,beleza_saude,beadbee30901a7f61d031b6b686095ad,55,14,25.45
3,None,1ca7077d890b907f89be8c954a02686a,68,17,25.00
4,beleza_saude,06a2c3af7b3aee5d69171b0e14f0ee87,389,90,23.14
5,automotivo,712e6ed8aa4aa1fa65dab41fed5737e4,77,17,22.08
6,fashion_bolsas_e_acessorios,e5a3438891c0bfdb9394643f95273d8e,142,31,21.83
7,alimentos,d13e50eaa47b4cbe9eb81465865d8cfc,55,12,21.82
8,moveis_decoracao,2a261b5b644fa05f4f2700eb93544f2c,51,10,19.61
9,informatica_acessorios,88460e8ebdecbfecb5f9601833981930,246,48,19.51


In [36]:
con.execute("""
CREATE OR REPLACE TABLE zip_coordinates AS
SELECT
    geolocation_zip_code_prefix AS zip_prefix,
    AVG(geolocation_lat) AS latitude,
    AVG(geolocation_lng) AS longitude
FROM geolocation
GROUP BY geolocation_zip_code_prefix;
""")

In [37]:
con.execute("""
WITH delivery_data AS (

    SELECT
        o.order_id,
        o.customer_id,
        oi.seller_id,

        c.customer_zip_code_prefix AS customer_zip,
        s.seller_zip_code_prefix AS seller_zip,

        cz.latitude AS customer_lat,
        cz.longitude AS customer_lng,

        sz.latitude AS seller_lat,
        sz.longitude AS seller_lng,

        CASE
            WHEN o.order_delivered_customer_date
                 > o.order_estimated_delivery_date
            THEN 1
            ELSE 0
        END AS is_late

    FROM orders o

    JOIN customers c
        ON o.customer_id = c.customer_id

    JOIN order_items oi
        ON o.order_id = oi.order_id

    JOIN sellers s
        ON oi.seller_id = s.seller_id

    LEFT JOIN zip_coordinates cz
        ON c.customer_zip_code_prefix = cz.zip_prefix

    LEFT JOIN zip_coordinates sz
        ON s.seller_zip_code_prefix = sz.zip_prefix

    WHERE o.order_status = 'delivered'
),

distance_data AS (

    SELECT
        *,
        6371 * 2 * ASIN(
            SQRT(
                POWER(
                    SIN(RADIANS(customer_lat - seller_lat) / 2),
                    2
                )
                +
                COS(RADIANS(seller_lat))
                * COS(RADIANS(customer_lat))
                * POWER(
                    SIN(RADIANS(customer_lng - seller_lng) / 2),
                    2
                )
            )
        ) AS distance_km

    FROM delivery_data

    WHERE customer_lat IS NOT NULL
      AND seller_lat IS NOT NULL
)

SELECT
    CASE
        WHEN distance_km < 100 THEN '<100 km'
        WHEN distance_km < 300 THEN '100–300 km'
        WHEN distance_km < 600 THEN '300–600 km'
        WHEN distance_km < 1000 THEN '600–1000 km'
        ELSE '1000+ km'
    END AS distance_band,

    COUNT(*) AS delivered_orders,

    SUM(is_late) AS late_orders,

    ROUND(
        100.0 * SUM(is_late) / COUNT(*),
        2
    ) AS late_rate

FROM distance_data

GROUP BY distance_band

ORDER BY
    CASE distance_band
        WHEN '<100 km' THEN 1
        WHEN '100–300 km' THEN 2
        WHEN '300–600 km' THEN 3
        WHEN '600–1000 km' THEN 4
        WHEN '1000+ km' THEN 5
    END;
""").fetchdf()

,distance_band,delivered_orders,late_orders,late_rate
0,<100 km,20481,1309.0,6.39
1,100–300 km,15283,956.0,6.26
2,300–600 km,36167,2701.0,7.47
3,600–1000 km,20479,1679.0,8.20
4,1000+ km,17251,2017.0,11.69


In [38]:
con.execute("""
WITH delivery_data AS (

    SELECT
        o.order_id,
        oi.seller_id,

        c.customer_zip_code_prefix AS customer_zip,
        s.seller_zip_code_prefix AS seller_zip,

        cz.latitude AS customer_lat,
        cz.longitude AS customer_lng,

        sz.latitude AS seller_lat,
        sz.longitude AS seller_lng,

        CASE
            WHEN o.order_delivered_customer_date
                 > o.order_estimated_delivery_date
            THEN 1
            ELSE 0
        END AS is_late

    FROM orders o

    JOIN customers c
        ON o.customer_id = c.customer_id

    JOIN order_items oi
        ON o.order_id = oi.order_id

    JOIN sellers s
        ON oi.seller_id = s.seller_id

    LEFT JOIN zip_coordinates cz
        ON c.customer_zip_code_prefix = cz.zip_prefix

    LEFT JOIN zip_coordinates sz
        ON s.seller_zip_code_prefix = sz.zip_prefix

    WHERE o.order_status = 'delivered'
),

distance_data AS (

    SELECT
        *,

        6371 * 2 * ASIN(
            SQRT(
                POWER(
                    SIN(RADIANS(customer_lat - seller_lat) / 2),
                    2
                )
                +
                COS(RADIANS(seller_lat))
                * COS(RADIANS(customer_lat))
                * POWER(
                    SIN(RADIANS(customer_lng - seller_lng) / 2),
                    2
                )
            )
        ) AS distance_km

    FROM delivery_data

    WHERE customer_lat IS NOT NULL
      AND seller_lat IS NOT NULL
),

segmented AS (

    SELECT
        seller_id,

        CASE
            WHEN distance_km < 300 THEN '<300 km'
            WHEN distance_km < 600 THEN '300–600 km'
            WHEN distance_km < 1000 THEN '600–1000 km'
            ELSE '1000+ km'
        END AS distance_band,

        COUNT(*) AS delivered_orders,

        SUM(is_late) AS late_orders

    FROM distance_data

    GROUP BY
        seller_id,
        distance_band

)

SELECT
    seller_id,
    distance_band,
    delivered_orders,
    late_orders,

    ROUND(
        100.0 * late_orders / delivered_orders,
        2
    ) AS late_rate

FROM segmented

WHERE delivered_orders >= 50

ORDER BY late_rate DESC

LIMIT 30;
""").fetchdf()

,seller_id,distance_band,delivered_orders,late_orders,late_rate
0,2c9e548be18521d1c43cde1c582c6de8,300–600 km,51,24.0,47.06
1,88460e8ebdecbfecb5f9601833981930,600–1000 km,109,34.0,31.19
2,7e1fb0a3ebfb01ffb3a7dae98bf3238d,1000+ km,50,14.0,28.00
3,d20b021d3efdf267a402c402a48ea64b,300–600 km,55,15.0,27.27
4,7aa4334be125fcdd2ba64b3180029f14,300–600 km,56,15.0,26.79
5,8160255418d5aaa7dbdc9f4c64ebda44,600–1000 km,81,20.0,24.69
6,b33e7c55446eabf8fe1a42d037ac7d6d,300–600 km,55,13.0,23.64
7,06a2c3af7b3aee5d69171b0e14f0ee87,1000+ km,363,85.0,23.42
8,77530e9772f57a62c906e1c21538ab82,1000+ km,73,17.0,23.29
9,e5a3438891c0bfdb9394643f95273d8e,300–600 km,57,13.0,22.81


In [39]:
con.execute("""
SELECT
    CASE
        WHEN p.product_weight_g < 1000 THEN '<1 kg'
        WHEN p.product_weight_g < 3000 THEN '1–3 kg'
        WHEN p.product_weight_g < 5000 THEN '3–5 kg'
        WHEN p.product_weight_g < 10000 THEN '5–10 kg'
        ELSE '10+ kg'
    END AS weight_band,

    COUNT(DISTINCT o.order_id) AS delivered_orders,

    COUNT(DISTINCT CASE
        WHEN o.order_delivered_customer_date
             > o.order_estimated_delivery_date
        THEN o.order_id
    END) AS late_orders,

    ROUND(
        100.0 *
        COUNT(DISTINCT CASE
            WHEN o.order_delivered_customer_date
                 > o.order_estimated_delivery_date
            THEN o.order_id
        END)
        / COUNT(DISTINCT o.order_id),
        2
    ) AS late_rate

FROM orders o

JOIN order_items oi
    ON o.order_id = oi.order_id

JOIN products p
    ON oi.product_id = p.product_id

WHERE o.order_status = 'delivered'

GROUP BY weight_band

ORDER BY
    CASE weight_band
        WHEN '<1 kg' THEN 1
        WHEN '1–3 kg' THEN 2
        WHEN '3–5 kg' THEN 3
        WHEN '5–10 kg' THEN 4
        WHEN '10+ kg' THEN 5
    END;
""").fetchdf()

,weight_band,delivered_orders,late_orders,late_rate
0,<1 kg,57301,4424,7.72
1,1–3 kg,23188,1903,8.21
2,3–5 kg,4642,425,9.16
3,5–10 kg,7722,613,7.94
4,10+ kg,4588,497,10.83


In [40]:
con.execute("""
SELECT

    CASE
        WHEN DATE_DIFF(
            'day',
            order_purchase_timestamp,
            order_delivered_carrier_date
        ) <= 2 THEN '0–2 days'

        WHEN DATE_DIFF(
            'day',
            order_purchase_timestamp,
            order_delivered_carrier_date
        ) <= 5 THEN '3–5 days'

        WHEN DATE_DIFF(
            'day',
            order_purchase_timestamp,
            order_delivered_carrier_date
        ) <= 10 THEN '6–10 days'

        ELSE '10+ days'
    END AS seller_handling_time,

    COUNT(*) AS delivered_orders,

    COUNT(
        CASE
            WHEN order_delivered_customer_date
                 > order_estimated_delivery_date
            THEN 1
        END
    ) AS late_orders,

    ROUND(
        100.0 *
        COUNT(
            CASE
                WHEN order_delivered_customer_date
                     > order_estimated_delivery_date
                THEN 1
            END
        )
        / COUNT(*),
        2
    ) AS late_rate

FROM orders

WHERE order_status = 'delivered'

  AND order_delivered_carrier_date IS NOT NULL

  AND order_delivered_customer_date IS NOT NULL

GROUP BY seller_handling_time

ORDER BY
    CASE seller_handling_time
        WHEN '0–2 days' THEN 1
        WHEN '3–5 days' THEN 2
        WHEN '6–10 days' THEN 3
        WHEN '10+ days' THEN 4
    END;
""").fetchdf()

,seller_handling_time,delivered_orders,late_orders,late_rate
0,0–2 days,52249,2878,5.51
1,3–5 days,30430,2426,7.97
2,6–10 days,10388,1374,13.23
3,10+ days,3402,1147,33.72


In [41]:
con.execute("""
SELECT

    CASE
        WHEN DATE_DIFF(
            'day',
            order_delivered_carrier_date,
            order_delivered_customer_date
        ) <= 3 THEN '0–3 days'

        WHEN DATE_DIFF(
            'day',
            order_delivered_carrier_date,
            order_delivered_customer_date
        ) <= 7 THEN '4–7 days'

        WHEN DATE_DIFF(
            'day',
            order_delivered_carrier_date,
            order_delivered_customer_date
        ) <= 14 THEN '8–14 days'

        ELSE '15+ days'
    END AS carrier_transit_time,

    COUNT(*) AS delivered_orders,

    COUNT(
        CASE
            WHEN order_delivered_customer_date
                 > order_estimated_delivery_date
            THEN 1
        END
    ) AS late_orders,

    ROUND(
        100.0 *
        COUNT(
            CASE
                WHEN order_delivered_customer_date
                     > order_estimated_delivery_date
                THEN 1
            END
        )
        / COUNT(*),
        2
    ) AS late_rate

FROM orders

WHERE order_status = 'delivered'

  AND order_delivered_carrier_date IS NOT NULL

  AND order_delivered_customer_date IS NOT NULL

GROUP BY carrier_transit_time

ORDER BY
    CASE carrier_transit_time
        WHEN '0–3 days' THEN 1
        WHEN '4–7 days' THEN 2
        WHEN '8–14 days' THEN 3
        WHEN '15+ days' THEN 4
    END;
""").fetchdf()

,carrier_transit_time,delivered_orders,late_orders,late_rate
0,0–3 days,20550,489,2.38
1,4–7 days,31584,469,1.48
2,8–14 days,28111,843,3.00
3,15+ days,16224,6024,37.13


In [42]:
con.execute("""
WITH order_stages AS (

    SELECT
        order_id,

        DATE_DIFF(
            'day',
            order_purchase_timestamp,
            order_delivered_carrier_date
        ) AS seller_handling_days,

        DATE_DIFF(
            'day',
            order_delivered_carrier_date,
            order_delivered_customer_date
        ) AS carrier_transit_days,

        CASE
            WHEN order_delivered_customer_date > order_estimated_delivery_date
            THEN 1
            ELSE 0
        END AS is_late

    FROM orders

    WHERE order_status = 'delivered'
      AND order_delivered_carrier_date IS NOT NULL
      AND order_delivered_customer_date IS NOT NULL
      AND order_estimated_delivery_date IS NOT NULL
)

SELECT

    CASE

        WHEN seller_handling_days <= 5
             AND carrier_transit_days <= 7
        THEN 'Healthy'

        WHEN seller_handling_days > 5
             AND carrier_transit_days <= 7
        THEN 'Seller bottleneck'

        WHEN seller_handling_days <= 5
             AND carrier_transit_days > 7
        THEN 'Carrier bottleneck'

        WHEN seller_handling_days > 5
             AND carrier_transit_days > 7
        THEN 'Both bottlenecks'

    END AS bottleneck_type,

    COUNT(*) AS orders,

    SUM(is_late) AS late_orders,

    ROUND(
        100.0 * SUM(is_late) / COUNT(*),
        2
    ) AS late_rate

FROM order_stages

GROUP BY bottleneck_type

ORDER BY late_rate DESC;
""").fetchdf()

,bottleneck_type,orders,late_orders,late_rate
0,Both bottlenecks,6706,1915.0,28.56
1,Carrier bottleneck,37629,4952.0,13.16
2,Seller bottleneck,7084,606.0,8.55
3,Healthy,45050,352.0,0.78


In [43]:
con.execute("""
SELECT
    COALESCE(p.product_category_name, 'Unknown') AS category,

    COUNT(DISTINCT o.order_id) AS orders,

    COUNT(DISTINCT CASE
        WHEN o.order_delivered_customer_date
             > o.order_estimated_delivery_date
        THEN o.order_id
    END) AS late_orders,

    ROUND(
        100.0 *
        COUNT(DISTINCT CASE
            WHEN o.order_delivered_customer_date
                 > o.order_estimated_delivery_date
            THEN o.order_id
        END)
        / COUNT(DISTINCT o.order_id),
        2
    ) AS late_rate

FROM orders o

JOIN order_items oi
    ON o.order_id = oi.order_id

JOIN products p
    ON oi.product_id = p.product_id

WHERE o.order_status = 'delivered'

GROUP BY category

HAVING COUNT(DISTINCT o.order_id) >= 1000

ORDER BY late_orders DESC;
""").fetchdf()

,category,orders,late_orders,late_rate
0,cama_mesa_banho,9272,811,8.75
1,beleza_saude,8647,775,8.96
2,esporte_lazer,7530,584,7.76
3,moveis_decoracao,6307,535,8.48
4,informatica_acessorios,6530,503,7.70
5,relogios_presentes,5495,468,8.52
6,utilidades_domesticas,5743,399,6.95
7,telefonia,4093,349,8.53
8,automotivo,3810,328,8.61
9,brinquedos,3804,286,7.52


In [44]:
con.execute("""
SELECT

    CASE
        WHEN o.order_delivered_customer_date
             > o.order_estimated_delivery_date
        THEN 'Late'
        ELSE 'Early / On Time'
    END AS delivery_performance,

    COUNT(DISTINCT o.order_id) AS orders,

    ROUND(AVG(r.review_score), 2) AS avg_review_score

FROM orders o

JOIN order_reviews r
    ON o.order_id = r.order_id

WHERE o.order_status = 'delivered'

GROUP BY delivery_performance

ORDER BY delivery_performance;
""").fetchdf()

,delivery_performance,orders,avg_review_score
0,Early / On Time,88171,4.29
1,Late,7661,2.57


In [45]:
con.execute("""
WITH order_value AS (

    SELECT
        o.order_id,

        SUM(oi.price + oi.freight_value) AS order_value,

        CASE
            WHEN o.order_delivered_customer_date
                 > o.order_estimated_delivery_date
            THEN 1
            ELSE 0
        END AS is_late

    FROM orders o

    JOIN order_items oi
        ON o.order_id = oi.order_id

    WHERE o.order_status = 'delivered'

      AND o.order_delivered_customer_date IS NOT NULL

      AND o.order_estimated_delivery_date IS NOT NULL

    GROUP BY
        o.order_id,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date
)

SELECT

    CASE
        WHEN order_value < 100 THEN '<100'
        WHEN order_value < 250 THEN '100–250'
        WHEN order_value < 500 THEN '250–500'
        WHEN order_value < 1000 THEN '500–1000'
        ELSE '1000+'
    END AS order_value_band,

    COUNT(*) AS orders,

    SUM(is_late) AS late_orders,

    ROUND(
        100.0 * SUM(is_late) / COUNT(*),
        2
    ) AS late_rate,

    ROUND(AVG(order_value), 2) AS avg_order_value

FROM order_value

GROUP BY order_value_band

ORDER BY
    CASE order_value_band
        WHEN '<100' THEN 1
        WHEN '100–250' THEN 2
        WHEN '250–500' THEN 3
        WHEN '500–1000' THEN 4
        WHEN '1000+' THEN 5
    END;
""").fetchdf()

,order_value_band,orders,late_orders,late_rate,avg_order_value
0,<100,45924,3478.0,7.57,60.22
1,100–250,37396,3123.0,8.35,155.69
2,250–500,9076,839.0,9.24,337.37
3,500–1000,2967,277.0,9.34,680.50
4,1000+,1107,109.0,9.85,1580.22


In [46]:
con.execute("""
WITH order_value AS (

    SELECT
        o.order_id,

        SUM(oi.price + oi.freight_value) AS order_value,

        CASE
            WHEN o.order_delivered_customer_date
                 > o.order_estimated_delivery_date
            THEN 1
            ELSE 0
        END AS is_late

    FROM orders o

    JOIN order_items oi
        ON o.order_id = oi.order_id

    WHERE o.order_status = 'delivered'

      AND o.order_delivered_customer_date IS NOT NULL

      AND o.order_estimated_delivery_date IS NOT NULL

    GROUP BY
        o.order_id,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date
)

SELECT

    COUNT(*) AS delivered_orders,

    SUM(is_late) AS late_orders,

    ROUND(
        100.0 * SUM(is_late) / COUNT(*),
        2
    ) AS late_rate,

    ROUND(
        SUM(order_value),
        2
    ) AS total_order_value,

    ROUND(
        SUM(
            CASE
                WHEN is_late = 1
                THEN order_value
                ELSE 0
            END
        ),
        2
    ) AS late_order_value

FROM order_value;
""").fetchdf()

,delivered_orders,late_orders,late_rate,total_order_value,late_order_value
0,96470,7826.0,8.11,15418394.83,1351624.96


In [47]:
con.execute("""
WITH order_risk AS (

    SELECT

        order_id,

        DATE_DIFF(
            'day',
            order_purchase_timestamp,
            order_delivered_carrier_date
        ) AS seller_handling_days,

        DATE_DIFF(
            'day',
            order_delivered_carrier_date,
            order_delivered_customer_date
        ) AS carrier_transit_days,

        CASE
            WHEN order_delivered_customer_date
                 > order_estimated_delivery_date
            THEN 1
            ELSE 0
        END AS is_late

    FROM orders

    WHERE order_status = 'delivered'

      AND order_delivered_carrier_date IS NOT NULL

      AND order_delivered_customer_date IS NOT NULL

      AND order_estimated_delivery_date IS NOT NULL
)

SELECT

    CASE

        WHEN seller_handling_days <= 5
             AND carrier_transit_days <= 7
        THEN 'Low risk'

        WHEN seller_handling_days > 5
             AND carrier_transit_days <= 7
        THEN 'Seller risk'

        WHEN seller_handling_days <= 5
             AND carrier_transit_days > 7
        THEN 'Carrier risk'

        ELSE 'Critical'

    END AS risk_group,

    COUNT(*) AS orders,

    SUM(is_late) AS late_orders,

    ROUND(
        100.0 * SUM(is_late) / COUNT(*),
        2
    ) AS late_rate

FROM order_risk

GROUP BY risk_group

ORDER BY late_rate DESC;
""").fetchdf()

,risk_group,orders,late_orders,late_rate
0,Critical,6706,1915.0,28.56
1,Carrier risk,37629,4952.0,13.16
2,Seller risk,7084,606.0,8.55
3,Low risk,45050,352.0,0.78


In [48]:
con.execute("""
WITH order_risk AS (

    SELECT

        order_id,

        DATE_DIFF(
            'day',
            order_purchase_timestamp,
            order_delivered_carrier_date
        ) AS seller_handling_days,

        DATE_DIFF(
            'day',
            order_delivered_carrier_date,
            order_delivered_customer_date
        ) AS carrier_transit_days,

        CASE
            WHEN order_delivered_customer_date
                 > order_estimated_delivery_date
            THEN 1
            ELSE 0
        END AS is_late

    FROM orders

    WHERE order_status = 'delivered'

      AND order_delivered_carrier_date IS NOT NULL

      AND order_delivered_customer_date IS NOT NULL

      AND order_estimated_delivery_date IS NOT NULL
)

SELECT

    COUNT(*) AS total_orders,

    SUM(
        CASE
            WHEN seller_handling_days > 5
            THEN 1
            ELSE 0
        END
    ) AS seller_risk_orders,

    SUM(
        CASE
            WHEN carrier_transit_days > 7
            THEN 1
            ELSE 0
        END
    ) AS carrier_risk_orders,

    SUM(
        CASE
            WHEN seller_handling_days > 5
                 OR carrier_transit_days > 7
            THEN 1
            ELSE 0
        END
    ) AS any_risk_orders,

    SUM(is_late) AS actual_late_orders

FROM order_risk;
""").fetchdf()

,total_orders,seller_risk_orders,carrier_risk_orders,any_risk_orders,actual_late_orders
0,96469,13790.0,44335.0,51419.0,7825.0


In [49]:
con.execute("""
WITH order_risk AS (

    SELECT

        order_id,

        DATE_DIFF(
            'day',
            order_purchase_timestamp,
            order_delivered_carrier_date
        ) AS seller_days,

        DATE_DIFF(
            'day',
            order_delivered_carrier_date,
            order_delivered_customer_date
        ) AS carrier_days,

        CASE
            WHEN order_delivered_customer_date
                 > order_estimated_delivery_date
            THEN 1
            ELSE 0
        END AS is_late

    FROM orders

    WHERE order_status = 'delivered'

      AND order_delivered_carrier_date IS NOT NULL

      AND order_delivered_customer_date IS NOT NULL

      AND order_estimated_delivery_date IS NOT NULL
),

scored AS (

    SELECT

        *,

        CASE
            WHEN seller_days <= 2 THEN 0
            WHEN seller_days <= 5 THEN 1
            WHEN seller_days <= 10 THEN 2
            ELSE 3
        END

        +

        CASE
            WHEN carrier_days <= 7 THEN 0
            WHEN carrier_days <= 14 THEN 1
            ELSE 3
        END

        AS risk_score

    FROM order_risk
)

SELECT

    risk_score,

    COUNT(*) AS orders,

    SUM(is_late) AS late_orders,

    ROUND(
        100.0 * SUM(is_late) / COUNT(*),
        2
    ) AS late_rate

FROM scored

GROUP BY risk_score

ORDER BY risk_score;
""").fetchdf()

,risk_score,orders,late_orders,late_rate
0,0,29276,160.0,0.55
1,1,30657,375.0,1.22
2,2,14377,372.0,2.59
3,3,13026,3119.0,23.94
4,4,6591,2384.0,36.17
5,5,1910,972.0,50.89
6,6,632,443.0,70.09


In [50]:
con.execute("""
WITH orders_stage AS (
    SELECT
        order_id,
        order_purchase_timestamp,
        order_delivered_carrier_date,
        order_delivered_customer_date,
        order_estimated_delivery_date,

        DATE_DIFF(
            'day',
            order_purchase_timestamp,
            order_delivered_carrier_date
        ) AS seller_handling_days

    FROM orders
    WHERE order_status = 'delivered'
      AND order_delivered_carrier_date IS NOT NULL
      AND order_delivered_customer_date IS NOT NULL
      AND order_estimated_delivery_date IS NOT NULL
),

risk_signal AS (
    SELECT
        *,
        CASE
            WHEN seller_handling_days <= 2 THEN 'Low'
            WHEN seller_handling_days <= 5 THEN 'Normal'
            WHEN seller_handling_days <= 10 THEN 'Elevated'
            ELSE 'Critical'
        END AS seller_risk,

        CASE
            WHEN order_delivered_customer_date > order_estimated_delivery_date
            THEN 1
            ELSE 0
        END AS is_late

    FROM orders_stage
)

SELECT
    seller_risk,
    COUNT(*) AS orders,
    SUM(is_late) AS late_orders,
    ROUND(
        100.0 * SUM(is_late) / COUNT(*),
        2
    ) AS late_rate
FROM risk_signal
GROUP BY seller_risk
ORDER BY
    CASE seller_risk
        WHEN 'Low' THEN 1
        WHEN 'Normal' THEN 2
        WHEN 'Elevated' THEN 3
        WHEN 'Critical' THEN 4
    END;
""").fetchdf()

,seller_risk,orders,late_orders,late_rate
0,Low,52249,2878.0,5.51
1,Normal,30430,2426.0,7.97
2,Elevated,10388,1374.0,13.23
3,Critical,3402,1147.0,33.72


In [51]:
con.execute("""
WITH order_stages AS (

    SELECT
        order_id,

        DATE_DIFF(
            'day',
            order_purchase_timestamp,
            order_delivered_carrier_date
        ) AS seller_days,

        DATE_DIFF(
            'day',
            order_delivered_carrier_date,
            order_delivered_customer_date
        ) AS carrier_days,

        CASE
            WHEN order_delivered_customer_date > order_estimated_delivery_date
            THEN 1
            ELSE 0
        END AS is_late

    FROM orders

    WHERE order_status = 'delivered'
      AND order_delivered_carrier_date IS NOT NULL
      AND order_delivered_customer_date IS NOT NULL
      AND order_estimated_delivery_date IS NOT NULL
),

bottleneck AS (

    SELECT
        *,

        CASE

            WHEN seller_days > 5 AND carrier_days > 7
                THEN 'Both'

            WHEN seller_days > 5
                THEN 'Seller'

            WHEN carrier_days > 7
                THEN 'Carrier'

            ELSE 'Healthy'

        END AS bottleneck

    FROM order_stages
)

SELECT
    bottleneck,
    COUNT(*) AS orders,
    SUM(is_late) AS late_orders,

    ROUND(
        100.0 * SUM(is_late) / COUNT(*),
        2
    ) AS late_rate

FROM bottleneck

GROUP BY bottleneck

ORDER BY late_rate DESC;
""").fetchdf()

,bottleneck,orders,late_orders,late_rate
0,Both,6706,1915.0,28.56
1,Carrier,37629,4952.0,13.16
2,Seller,7084,606.0,8.55
3,Healthy,45050,352.0,0.78


In [52]:
con.execute("""
WITH order_values AS (

    SELECT
        order_id,
        SUM(price + freight_value) AS order_value
    FROM order_items
    GROUP BY order_id

),

order_stages AS (

    SELECT
        order_id,

        DATE_DIFF(
            'day',
            order_purchase_timestamp,
            order_delivered_carrier_date
        ) AS seller_days,

        DATE_DIFF(
            'day',
            order_delivered_carrier_date,
            order_delivered_customer_date
        ) AS carrier_days,

        CASE
            WHEN order_delivered_customer_date > order_estimated_delivery_date
            THEN 1
            ELSE 0
        END AS is_late

    FROM orders

    WHERE order_status = 'delivered'
      AND order_delivered_carrier_date IS NOT NULL
      AND order_delivered_customer_date IS NOT NULL
      AND order_estimated_delivery_date IS NOT NULL
),

bottleneck AS (

    SELECT
        *,
        CASE
            WHEN seller_days > 5 AND carrier_days > 7
                THEN 'Both'

            WHEN seller_days > 5
                THEN 'Seller'

            WHEN carrier_days > 7
                THEN 'Carrier'

            ELSE 'Healthy'
        END AS bottleneck

    FROM order_stages
)

SELECT

    b.bottleneck,

    COUNT(*) AS orders,

    SUM(b.is_late) AS late_orders,

    ROUND(
        100.0 * SUM(b.is_late) / COUNT(*),
        2
    ) AS late_rate,

    ROUND(
        SUM(v.order_value),
        2
    ) AS total_order_value,

    ROUND(
        SUM(
            CASE
                WHEN b.is_late = 1
                THEN v.order_value
                ELSE 0
            END
        ),
        2
    ) AS late_order_value

FROM bottleneck b

JOIN order_values v
    ON b.order_id = v.order_id

GROUP BY b.bottleneck

ORDER BY late_rate DESC;
""").fetchdf()

,bottleneck,orders,late_orders,late_rate,total_order_value,late_order_value
0,Both,6706,1915.0,28.56,1324356.19,390026.35
1,Carrier,37629,4952.0,13.16,6185449.64,797063.86
2,Seller,7084,606.0,8.55,1332877.40,121239.29
3,Healthy,45050,352.0,0.78,6575517.62,43101.48


In [53]:
order_monitor = con.execute("""
WITH order_values AS (
    SELECT
        order_id,
        SUM(price + freight_value) AS order_value
    FROM order_items
    GROUP BY order_id
),

review_data AS (
    SELECT
        order_id,
        AVG(review_score) AS review_score
    FROM order_reviews
    GROUP BY order_id
),

order_monitor AS (
    SELECT
        o.order_id,
        c.customer_unique_id,

        CAST(o.order_purchase_timestamp AS DATE) AS order_date,

        DATE_DIFF(
            'day',
            o.order_purchase_timestamp,
            o.order_delivered_carrier_date
        ) AS seller_days,

        DATE_DIFF(
            'day',
            o.order_delivered_carrier_date,
            o.order_delivered_customer_date
        ) AS carrier_days,

        DATE_DIFF(
            'day',
            o.order_purchase_timestamp,
            o.order_estimated_delivery_date
        ) AS promised_days,

        v.order_value,

        r.review_score,

        CASE
            WHEN o.order_delivered_customer_date >
                 o.order_estimated_delivery_date
            THEN 'Late'
            ELSE 'On Time'
        END AS delivery_status

    FROM orders o

    JOIN customers c
        ON o.customer_id = c.customer_id

    JOIN order_values v
        ON o.order_id = v.order_id

    LEFT JOIN review_data r
        ON o.order_id = r.order_id

    WHERE o.order_status = 'delivered'
      AND o.order_delivered_carrier_date IS NOT NULL
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
),

classified AS (
    SELECT
        *,

        CASE
            WHEN seller_days > 5 AND carrier_days > 7
                THEN 'Both'
            WHEN seller_days > 5
                THEN 'Seller'
            WHEN carrier_days > 7
                THEN 'Carrier'
            ELSE 'Healthy'
        END AS bottleneck,

        CASE
            WHEN seller_days > 10 OR carrier_days > 14
                THEN 'Critical'

            WHEN seller_days > 5 OR carrier_days > 7
                THEN 'Elevated'

            ELSE 'Low'
        END AS risk_level

    FROM order_monitor
)

SELECT
    *,

    CASE
        WHEN bottleneck = 'Both'
            THEN 'Priority operations intervention'

        WHEN bottleneck = 'Seller'
            THEN 'Seller follow-up'

        WHEN bottleneck = 'Carrier'
            THEN 'Carrier escalation + ETA review'

        ELSE 'Continue monitoring'
    END AS recommended_action

FROM classified
ORDER BY
    CASE risk_level
        WHEN 'Critical' THEN 1
        WHEN 'Elevated' THEN 2
        ELSE 3
    END,
    order_date;
""").fetchdf()

order_monitor.head()

,order_id,customer_unique_id,order_date,seller_days,carrier_days,promised_days,order_value,review_score,delivery_status,bottleneck,risk_level,recommended_action
0,bfbd0f9bdef84302105ad712db648a6c,830d5b7aaa3b6f1e9ad63703bec97d23,2016-09-15,53,2,19,143.46,1.0,Late,Seller,Critical,Seller follow-up
1,ef1b29b591d31d57c0d7337460dd83c9,10e89fd8e5c745f81bec101207ba4d7d,2016-10-03,18,11,53,92.27,1.0,On Time,Both,Critical,Priority operations intervention
2,a41c8759fbe7aab36ea07e038b2d4465,61db744d2f835035a5625b59350c6b63,2016-10-03,22,9,57,53.73,3.0,On Time,Both,Critical,Priority operations intervention
3,3b697a20d9e427646d92567910af6d57,32ea3bdedab835c3aa6cb68ce66565ef,2016-10-03,20,3,24,45.46,4.0,On Time,Seller,Critical,Seller follow-up
4,ae8a60e4b03c5a4ba9ca0672c164b181,7390ed59fa1febbfda31a80b4318c8cb,2016-10-03,27,4,59,154.57,5.0,On Time,Seller,Critical,Seller follow-up


In [54]:
# Export our order-level monitoring dataset
order_monitor.to_csv("olist_fulfillment_risk_monitor.csv", index=False)

print("Saved!")
print("Rows:", len(order_monitor))
print("Columns:", len(order_monitor.columns))

Saved!
Rows: 96469
Columns: 12


In [55]:
risk_summary = con.execute("""
WITH order_values AS (
    SELECT
        order_id,
        SUM(price + freight_value) AS order_value
    FROM order_items
    GROUP BY order_id
),

base AS (
    SELECT
        o.order_id,

        DATE_DIFF(
            'day',
            o.order_purchase_timestamp,
            o.order_delivered_carrier_date
        ) AS seller_days,

        DATE_DIFF(
            'day',
            o.order_delivered_carrier_date,
            o.order_delivered_customer_date
        ) AS carrier_days,

        v.order_value,

        CASE
            WHEN o.order_delivered_customer_date >
                 o.order_estimated_delivery_date
            THEN 1
            ELSE 0
        END AS is_late

    FROM orders o

    JOIN order_values v
        ON o.order_id = v.order_id

    WHERE o.order_status = 'delivered'
      AND o.order_delivered_carrier_date IS NOT NULL
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
),

classified AS (
    SELECT
        *,

        CASE
            WHEN seller_days > 5 AND carrier_days > 7
                THEN 'Both'
            WHEN seller_days > 5
                THEN 'Seller'
            WHEN carrier_days > 7
                THEN 'Carrier'
            ELSE 'Healthy'
        END AS bottleneck,

        CASE
            WHEN seller_days > 10 OR carrier_days > 14
                THEN 'Critical'
            WHEN seller_days > 5 OR carrier_days > 7
                THEN 'Elevated'
            ELSE 'Low'
        END AS risk_level

    FROM base
)

SELECT
    risk_level,
    bottleneck,

    COUNT(*) AS orders,

    SUM(is_late) AS late_orders,

    ROUND(
        100.0 * SUM(is_late) / COUNT(*),
        2
    ) AS late_rate,

    ROUND(SUM(order_value), 2)
        AS total_order_value,

    ROUND(
        SUM(
            CASE
                WHEN is_late = 1
                THEN order_value
                ELSE 0
            END
        ),
        2
    ) AS late_order_value

FROM classified

GROUP BY
    risk_level,
    bottleneck

ORDER BY
    CASE risk_level
        WHEN 'Critical' THEN 1
        WHEN 'Elevated' THEN 2
        ELSE 3
    END,
    late_rate DESC;
""").fetchdf()

risk_summary

,risk_level,bottleneck,orders,late_orders,late_rate,total_order_value,late_order_value
0,Critical,Both,3541,1725.0,48.72,754379.83,354908.77
1,Critical,Carrier,13682,4609.0,33.69,2283847.35,749562.19
2,Critical,Seller,1771,394.0,22.25,417229.16,89642.12
3,Elevated,Both,3165,190.0,6.00,569976.36,35117.58
4,Elevated,Seller,5313,212.0,3.99,915648.24,31597.17
5,Elevated,Carrier,23947,343.0,1.43,3901602.29,47501.67
6,Low,Healthy,45050,352.0,0.78,6575517.62,43101.48


In [56]:
con.execute("""
WITH base AS (
    SELECT
        o.order_id,

        DATE_DIFF(
            'day',
            o.order_purchase_timestamp,
            o.order_delivered_carrier_date
        ) AS seller_days,

        DATE_DIFF(
            'day',
            o.order_delivered_carrier_date,
            o.order_delivered_customer_date
        ) AS carrier_days,

        CASE
            WHEN o.order_delivered_customer_date >
                 o.order_estimated_delivery_date
            THEN 1
            ELSE 0
        END AS is_late

    FROM orders o

    WHERE o.order_status = 'delivered'
      AND o.order_delivered_carrier_date IS NOT NULL
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
),

classified AS (
    SELECT
        *,
        CASE
            WHEN seller_days > 10
              OR carrier_days > 14
            THEN 1
            ELSE 0
        END AS alert
    FROM base
)

SELECT
    SUM(alert) AS flagged_orders,

    SUM(
        CASE
            WHEN alert = 1 AND is_late = 1
            THEN 1 ELSE 0
        END
    ) AS flagged_late_orders,

    SUM(is_late) AS total_late_orders,

    ROUND(
        100.0 *
        SUM(CASE WHEN alert = 1 AND is_late = 1 THEN 1 ELSE 0 END)
        / NULLIF(SUM(alert), 0),
        2
    ) AS precision_pct,

    ROUND(
        100.0 *
        SUM(CASE WHEN alert = 1 AND is_late = 1 THEN 1 ELSE 0 END)
        / NULLIF(SUM(is_late), 0),
        2
    ) AS recall_pct

FROM classified;
""").fetchdf()

,flagged_orders,flagged_late_orders,total_late_orders,precision_pct,recall_pct
0,18994.0,6728.0,7825.0,35.42,85.98


In [57]:
intervention_queue = order_monitor[
    order_monitor["risk_level"] == "Critical"
].copy()

intervention_queue = intervention_queue[
    [
        "order_id",
        "order_date",
        "seller_days",
        "carrier_days",
        "order_value",
        "bottleneck",
        "risk_level",
        "delivery_status",
        "recommended_action"
    ]
].sort_values(
    ["order_value", "seller_days", "carrier_days"],
    ascending=[False, False, False]
)

print("Orders requiring intervention:", len(intervention_queue))

intervention_queue.head(10)

Orders requiring intervention: 18994


,order_id,order_date,seller_days,carrier_days,order_value,bottleneck,risk_level,delivery_status,recommended_action
4086,03caa2c082116e1d31e67e9ae3700499,2017-09-29,11,7,13664.08,Seller,Critical,On Time,Seller follow-up
387,0812eb902a67711a1cb742b3cdaa65ae,2017-02-12,4,15,6929.31,Carrier,Critical,On Time,Carrier escalation + ETA review
5759,2cc9089445046817a7539d90805e6e5a,2017-11-24,12,7,6081.54,Seller,Critical,On Time,Seller follow-up
5336,d7a2c0c1ff66b314f3bf166fb4157fd4,2017-11-20,15,37,3184.55,Both,Critical,Late,Priority operations intervention
1045,f0da489592c62097d0034f139386cb32,2017-04-05,8,19,3009.53,Both,Critical,On Time,Priority operations intervention
14339,3c4a05391c2fcd152731f52de8fc4347,2018-03-18,1,20,2960.05,Carrier,Critical,On Time,Carrier escalation + ETA review
1768,794c69999774820e22c161f6dc4fef91,2017-05-09,20,11,2794.50,Both,Critical,On Time,Priority operations intervention
6374,877a74a0ee5c1f719f39f4f11d2fc19a,2017-11-27,1,21,2759.88,Carrier,Critical,On Time,Carrier escalation + ETA review
1060,da8be3bb62e9bf01e2e1a3bfd74ebd1a,2017-04-06,2,30,2751.24,Carrier,Critical,Late,Carrier escalation + ETA review
6858,3745e06e7f01e1652cfd4b9d31651820,2017-12-02,2,15,2734.66,Carrier,Critical,On Time,Carrier escalation + ETA review


In [58]:
intervention_queue["priority"] = intervention_queue.apply(
    lambda row:
        "P1" if row["bottleneck"] == "Both" and row["order_value"] >= 500
        else "P2" if row["order_value"] >= 500
        else "P3",
    axis=1
)

intervention_queue = intervention_queue.sort_values(
    ["priority", "order_value"],
    ascending=[True, False]
)

intervention_queue.head(10)


,order_id,order_date,seller_days,carrier_days,order_value,bottleneck,risk_level,delivery_status,recommended_action,priority
5336,d7a2c0c1ff66b314f3bf166fb4157fd4,2017-11-20,15,37,3184.55,Both,Critical,Late,Priority operations intervention,P1
1045,f0da489592c62097d0034f139386cb32,2017-04-05,8,19,3009.53,Both,Critical,On Time,Priority operations intervention,P1
1768,794c69999774820e22c161f6dc4fef91,2017-05-09,20,11,2794.50,Both,Critical,On Time,Priority operations intervention,P1
8204,bf205457ee84ab84c423d67b88239982,2017-12-21,18,18,2733.63,Both,Critical,Late,Priority operations intervention,P1
18935,6d0940a8f5fba47562bb14cd97dfd6da,2018-08-10,24,9,2455.12,Both,Critical,Late,Priority operations intervention,P1
17946,947ee6ab639791b5711558a7e55cf98e,2018-06-11,37,10,2223.12,Both,Critical,On Time,Priority operations intervention,P1
1021,8ae8b33f63d9924a2db9ec6c1e98fd35,2017-04-03,22,10,2111.57,Both,Critical,Late,Priority operations intervention,P1
12200,cfed507ac357129f750f05a0d7d71b15,2018-02-25,11,29,2091.33,Both,Critical,Late,Priority operations intervention,P1
2168,725cf8e9c24e679a8a5a32cb92c9ce1e,2017-06-08,15,13,2067.42,Both,Critical,On Time,Priority operations intervention,P1
3249,c819d9ce6576bb453e886c2e95aa5697,2017-08-16,30,10,2065.35,Both,Critical,On Time,Priority operations intervention,P1


In [59]:
# Export the two datasets for Google Sheets

intervention_queue.to_csv(
    "intervention_queue.csv",
    index=False
)

risk_summary.to_csv(
    "risk_summary.csv",
    index=False
)

print("Files created:")
print("1. intervention_queue.csv")
print("2. risk_summary.csv")

Files created:
1. intervention_queue.csv
2. risk_summary.csv
